In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cpu
False


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report
import numpy as np

# -------------------------
# 1. DEVICE
# -------------------------
device = torch.device("cpu")
print("Using device:", device)

# -------------------------
# 2. TRANSFORMS (LIGHT)
# -------------------------
transform = transforms.Compose([
    transforms.Resize((128, 128)),  # smaller = faster
    transforms.ToTensor()
])

# -------------------------
# 3. LOAD DATA
# -------------------------
dataset = datasets.ImageFolder("D:\Dia Foot\DFU\Classification Folder", transform=transform)

class_names = dataset.classes
print("Classes:", class_names)

# Split dataset (80 train / 20 val)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_data, val_data = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(val_data, batch_size=16, shuffle=False)

# -------------------------
# 4. SIMPLE CNN (FAST)
# -------------------------
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 64),
            nn.ReLU(),
            nn.Linear(64, 4)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

model = SimpleCNN().to(device)

# -------------------------
# 5. LOSS + OPTIMIZER
# -------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# -------------------------
# 6. TRAINING
# -------------------------
epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

# -------------------------
# 7. EVALUATION
# -------------------------
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("\nClassification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

# -------------------------
# 8. SAVE MODEL
# -------------------------
torch.save(model.state_dict(), "dfu_model_cpu.pth")
print("\nModel saved as dfu_model_cpu.pth")

<>:26: SyntaxWarning: invalid escape sequence '\D'
<>:26: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Admin\AppData\Local\Temp\ipykernel_2912\1980181199.py:26: SyntaxWarning: invalid escape sequence '\D'
  dataset = datasets.ImageFolder("D:\Dia Foot\DFU\Classification Folder", transform=transform)


Using device: cpu
Classes: ['Normal', 'high_risk', 'low_risk', 'moderate']
Epoch 1, Loss: 140.7068
Epoch 2, Loss: 99.3670
Epoch 3, Loss: 85.5897
Epoch 4, Loss: 72.3230
Epoch 5, Loss: 59.9946

Classification Report:

              precision    recall  f1-score   support

      Normal       0.91      0.96      0.94       109
   high_risk       0.63      0.78      0.70       111
    low_risk       0.87      0.65      0.75       150
    moderate       0.60      0.61      0.61       165

    accuracy                           0.73       535
   macro avg       0.75      0.75      0.75       535
weighted avg       0.74      0.73      0.73       535


Model saved as dfu_model_cpu.pth
